# İP5 — XGBoost (Gradient Boosting) + Adaptif AKTS

**Görev (Bölüm 10, Görev 6):** Adaptif AKTS Hesaplayan Tahmin Modeli Oluşturma
**Sorumlu:** Samet Koca | **Tarih:** 18 Temmuz 2026 (model güncellemesi: 20 Temmuz 2026)

**Girdi:** `data/processed/model_egitim.xlsx` (43 sütun) + `data/processed/master_bloom.xlsx` (K_profil, K_etkinlik)

**Çıktı:** `data/processed/master_akts_final.xlsx` — her ders için `model_tahmin_akts`, `kural_akts`, `adaptif_akts`, `sapma`, `karar`

**Model güncellemesi (20 Temmuz 2026):** Random Forest yerine, Optuna ile ayarlanmış XGBoost kullanılıyor — proje planının resmi metodolojisi (Bölüm 6.3) "Gradient Boosting" istiyor ve ayarlanmış XGBoost, MAE/RMSE/R²'nin üçünde de Random Forest'ı geçti (bkz. `XGBOOST_MODEL_SECIMI_GEREKCESI.md`).

Bu notebook, `ip5-adaptif-akts/src/ip5_adaptif_akts.py` pipeline'ını çalıştırır ve sonuçları görselleştirir.

### Formüller

$$kural\_akts = akts\_mevcut \times K\_etkinlik \times K\_profil \times K\_bilissel$$

$$adaptif\_akts = 0.70 \times rf\_tahmin\_akts + 0.30 \times kural\_akts$$

**Hedef sızıntısı önlemi:** `akts_mevcut` (AKTSKredi) ve `K_bilissel`/`K_profil`/`K_etkinlik` RF'in özellik kümesine DAHİL EDİLMEZ — sadece kural tabanlı bileşende kullanılır (Karar 2b).

In [1]:
import sys
from pathlib import Path

BASE = Path.cwd().resolve().parents[1]  # course-credits-system köküne çık
sys.path.insert(0, str(BASE / "ip5-adaptif-akts" / "src"))

import ip5_adaptif_akts as ip5

final_df = ip5.main()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


İP5 — XGBOOST (GRADIENT BOOSTING) + ADAPTİF AKTS

Yükleniyor: model_egitim.xlsx


  4133 satır, 43 sütun
Yükleniyor: master_bloom.xlsx


  4133 satır, 107 sütun

Ayarlanmış XGBoost parametreleri (xgboost_en_iyi_parametreler.json'ten): {'max_depth': 6, 'learning_rate': 0.04614126735633599, 'subsample': 0.9641437513503683, 'colsample_bytree': 0.6864936147132107, 'reg_alpha': 2.2536604744440076, 'reg_lambda': 1.8297509620765344, 'min_child_weight': 2, 'n_estimators': 1000}

Özellik sayısı (X): 39
Hariç tutulan sütunlar: ['Katalogid', 'DersAdi', 'AKTSKredi', 'K_bilissel']

── Model Kalitesi (ayrılmış %20 test seti) ──────────────
  MAE  : 0.8105 AKTS
  RMSE : 1.0824 AKTS
  R²   : 0.1896

5 katlı stratified (Fakülte) çapraz doğrulama ile out-of-fold tahmin üretiliyor...


  Fold 1/5 tamamlandı (827 ders tahmin edildi, best_iteration=47)
  Fold 2/5 tamamlandı (827 ders tahmin edildi, best_iteration=97)


  Fold 3/5 tamamlandı (827 ders tahmin edildi, best_iteration=514)
  Fold 4/5 tamamlandı (826 ders tahmin edildi, best_iteration=59)


  Fold 5/5 tamamlandı (826 ders tahmin edildi, best_iteration=454)

── Model Kalitesi (tüm veri, out-of-fold) ────────────────
  MAE  : 0.8863 AKTS
  RMSE : 1.5503 AKTS
  R²   : 0.1242



── En Önemli 10 Özellik (SHAP ortalama mutlak katkı) ─────
  derstur_Teorik              : 0.1835
  Fakulte_Güzel Sanatlar      : 0.1310
  bloom_avg_level             : 0.1063
  Fakulte_Fen-Edebiyat        : 0.0972
  n_etkinlik                  : 0.0685
  n_ogrenme_kazanim           : 0.0493
  bloom_max_level             : 0.0472
  has_proje                   : 0.0461
  alan_grubu_Fen-Sosyal       : 0.0396
  derstur_Teori+Uygulama      : 0.0347

── Karar Dağılımı (0.70/0.30 ağırlık, primary) ───────────
  Güçlü uyum        :  1929 ders  (46.7%)
  Kabul edilebilir  :  1544 ders  (37.4%)
  İncelenmeli       :   660 ders  (16.0%)

── Duyarlılık Analizi: Ağırlık Şemalarına Göre Ortalama |Sapma%| ──
  0.50/0.50  -> ort |sapma%|=14.18  incelenmeli oranı=9.85%
  0.60/0.40  -> ort |sapma%|=15.41  incelenmeli oranı=12.58%
  0.70/0.30  -> ort |sapma%|=16.94  incelenmeli oranı=15.97%
  0.80/0.20  -> ort |sapma%|=18.72  incelenmeli oranı=19.31%



Kaydedildi: /Users/sudenazguven/Desktop/akts/data/processed/master_akts_final.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-adaptif-akts/feature_importance.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-adaptif-akts/duyarlilik_analizi.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-adaptif-akts/IP5_SONUC_RAPORU.md


### Sonuçları İncele

In [2]:
final_df[['DersAdi', 'Fakulte', 'AKTSKredi', 'model_tahmin_akts', 'kural_akts', 'adaptif_akts', 'sapma_yuzde', 'karar']].sample(10, random_state=1)

,DersAdi,Fakulte,AKTSKredi,model_tahmin_akts,kural_akts,adaptif_akts,sapma_yuzde,karar
2534,Hidrolik Makinalar,Mühendislik,6,4.1643,5.9322,4.6947,-21.76,Kabul edilebilir
804,İleri Laboratuvar II,Fen-Edebiyat,5,4.8829,8.4139,5.9422,18.84,Kabul edilebilir
3486,Medikal İllüstrasyon II,Güzel Sanatlar,4,4.2502,4.2712,4.2565,6.41,Güçlü uyum
709,Elektrokimya,Fen-Edebiyat,5,4.7393,5.1907,4.8747,-2.51,Güçlü uyum
2950,Ayrık Matematik,Mühendislik,4,3.9781,4.3503,4.0898,2.24,Güçlü uyum
1690,Doğalgaz ve Sıhhi Tesisat,Teknoloji,4,4.0432,4.1003,4.0603,1.51,Güçlü uyum
829,Bitirme Çalışması,Fen-Edebiyat,1,3.4858,1.3100,2.8331,183.31,İncelenmeli
3613,Toplu Müzik Eğitimi 5,Güzel Sanatlar,3,3.0379,2.9661,3.0164,0.55,Güçlü uyum
1328,EĞLENCELİ ATLETİZM,Spor Bilimleri,4,4.5632,4.6327,4.5840,14.60,Kabul edilebilir
2610,Çözelti Termodinamiği,Mühendislik,4,4.1895,3.9548,4.1191,2.98,Güçlü uyum


In [3]:
import pandas as pd

importance_df = pd.read_excel(BASE / 'ip5-adaptif-akts' / 'feature_importance.xlsx')
importance_df.head(15)

,ozellik,ortalama_mutlak_shap_katkisi
0,derstur_Teorik,0.183454
1,Fakulte_Güzel Sanatlar,0.130980
2,bloom_avg_level,0.106349
3,Fakulte_Fen-Edebiyat,0.097159
4,n_etkinlik,0.068488
5,n_ogrenme_kazanim,0.049313
6,bloom_max_level,0.047152
7,has_proje,0.046122
8,alan_grubu_Fen-Sosyal,0.039556
9,derstur_Teori+Uygulama,0.034666


In [4]:
sensitivity_df = pd.read_excel(BASE / 'ip5-adaptif-akts' / 'duyarlilik_analizi.xlsx')
sensitivity_df

,agirlik,ort_mutlak_sapma_yuzde,incelenmeli_oran_yuzde
0,0.50 Model / 0.50 Kural,14.18,9.85
1,0.60 Model / 0.40 Kural,15.41,12.58
2,0.70 Model / 0.30 Kural,16.94,15.97
3,0.80 Model / 0.20 Kural,18.72,19.31


### Bulgu ve İP6'ya Devir

Model kalitesi (out-of-fold) düşük/negatif R² gösteriyor — bu beklenen bir durumdur: amaç mevcut AKTS'yi ezberlemek değil, ondan anlamlı biçimde sapan dersleri işaretlemektir ("iyi taklit paradoksu", README.md §15.5). Ayrıntılı rapor: `ip5-adaptif-akts/IP5_SONUC_RAPORU.md`.

Çıktı (`master_akts_final.xlsx`), İP6'da (Görev 8) tutarsızlık analizi ve hipotez testleri için girdi olarak kullanılır.